# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [3]:
!pip install -U langchain langchain-classic langchain-openai langchain-community tavily-python

     ---------------------------------------- 0.0/114.3 kB ? eta -:--:--
     -------------------------------------  112.6/114.3 kB 3.3 MB/s eta 0:00:01
     -------------------------------------- 114.3/114.3 kB 2.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
     - -------------------------------------- 0.1/1.0 MB ? eta -:--:--
     - -------------------------------------- 0.1/1.0 MB ? eta -:--:--
     - -------------------------------------- 0.1/1.0 MB ? eta -:--:--
     - -------------------------------------- 0.1/1.0 MB ? eta -:--:--
     - -------------------------------------- 0.1/1.0 MB ? eta -:--:--
     - -------------------------------------- 0.1/1.0 MB ? eta -:--:--
     --- ------------------------------------ 0.1/1.0 MB 294.4 kB/s eta 0:00:04
     --------- ------------------------------ 0.2/1.0 MB 627.5 kB/s eta 0:00:02
     --------- ------------------------------ 0.2/1.0 MB 628.1 kB/s eta 0:00:02
     ---------------- --------

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'C:\\Python310\\Scripts\\dotenv.exe' -> 'C:\\Python310\\Scripts\\dotenv.exe.deleteme'


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: C:\Python310\python.exe -m pip install --upgrade pip


In [1]:
import os
import warnings
import tiktoken

from dotenv import load_dotenv
from IPython.display import Markdown, display

from langchain_classic.agents import initialize_agent, AgentType
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_community.tools import TavilySearchResults

warnings.filterwarnings("ignore")

load_dotenv()

tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)

    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])
    return trimmed

# LangChain tool
tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Useful for searching the web for recent information."
    )
]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3,
    early_stopping_method="generate"
)

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [4]:
# Your prompt here
prompt_1 = """
You are a research assistant using real-time web search.

Task:
Investigate the latest advancements in generative AI applied to healthcare in 2025.

Requirements:
- Use web search to retrieve recent and reliable information.
- Focus on developments from 2025 where possible.
- Prioritise credible sources such as healthcare organisations, research institutions, medical technology companies, government/regulatory bodies, or reputable news sources.
- Do not rely only on prior model knowledge.
- Summarise the findings in clear Markdown.

Your response must include:

# Generative AI in Healthcare: Latest Advancements in 2025

## 1. Executive Summary
Briefly summarise the most important trends.

## 2. Key Advancements
List 5-7 major advancements. For each one, include:
- What the advancement is
- Why it matters
- Example use case or organisation if available

## 3. Main Application Areas
Cover areas such as:
- Clinical decision support
- Medical imaging
- Drug discovery
- Patient communication
- Administrative automation
- Personalised medicine

## 4. Benefits
Summarise the main benefits for patients, clinicians, and healthcare systems.

## 5. Risks and Challenges
Discuss risks such as hallucinations, patient safety, privacy, bias, regulation, and clinical validation.

## 6. Outlook
Briefly explain where generative AI in healthcare appears to be heading next.

Keep the answer structured, concise, and practical.
"""

In [5]:
response_1 = agent.run(prompt_1)
display(Markdown(response_1))




> Entering new AgentExecutor chain...
I need to gather the latest information on advancements in generative AI applied to healthcare in 2025. This requires a web search to find credible and recent sources that discuss these developments. 

Action: TavilySafeSearch  
Action Input: "latest advancements in generative AI healthcare 2025"  

Observation: [{'title': 'How Healthcare is Using Generative AI in 2025', 'url': 'https://www.nexgencloud.com/blog/case-studies/how-healthcare-is-using-generative-ai', 'content': 'Generative AI in Healthcare uses techniques like generative adversarial networks (GANs) and large language models (LLMs) to identify intricate patterns within vast amounts of data, such as medical images, electronic health records (EHRs) and patient health data. These models can generate new, contextually relevant information, helping medical professionals diagnose and treat patients more effectively. [...] Systems like AI-Rad Companion use natural language generation (NLG) m

The summary above outlines the latest advancements in generative AI applied to healthcare in 2025, highlighting key trends, advancements, application areas, benefits, risks, and future outlook.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [7]:
# Your prompt here
prompt_2 = """
Use TavilySafeSearch to identify top emerging AI startups in 2025 and their innovations.

Return a clean Markdown summary.

Requirements:
- Use web search.
- Focus on startups that are active, visible, or fast-growing in 2025.
- Prioritise credible sources such as startup databases, VC reports, company pages, reputable tech news, or AI industry reports.
- Do not rely only on prior model knowledge.
- Keep the answer concise and practical.
- Include sources as plain text names or URLs if available from search results.
- Keep the response under 800 words.

Structure the response exactly like this:

# Emerging AI Startups in 2025

## Executive Summary
Briefly summarise the main trends in the 2025 AI startup landscape.

## Top Emerging AI Startups

For each startup, include:
- **Company name:**
- **Main product or innovation:**
- **Why it matters:**
- **Sector:** 
- **Source:**

Include 6-8 startups maximum.

## Key Innovation Trends
List 4-5 common trends across these startups.

## Final Takeaway
Give a short conclusion on where the AI startup landscape appears to be heading in 2025.
"""


In [8]:
response_2 = agent.invoke({"input": prompt_2})
display(Markdown(response_2["output"]))




> Entering new AgentExecutor chain...
I need to search for information on the top emerging AI startups in 2025, focusing on their innovations and ensuring that the sources are credible. I will use the TavilySafeSearch tool to find this information.

Action: TavilySafeSearch  
Action Input: "top emerging AI startups 2025 innovations"  

Observation: [{'title': 'The 10 Hottest AI Startup Companies Of 2025 (So Far) - CRN', 'url': 'https://www.crn.com/news/ai/2025/the-10-hottest-ai-startup-companies-of-2025-so-far', 'content': 'Many startups have already become mainstream AI providers around large language models (LLMs), customer support agents and code generation.\n\nOther AI startups like AI Squared, Morphos AI and Writer are focused on driving innovation around integrating AI directly into business applications, AI cost optimization offerings or platforms for building agents.\n\nRelated: [How AWS And Google Cloud Create Agentic AI Opportunities For Partners]\n\nIT research firm Gartne

OutputParserException: Could not parse LLM output: `# Emerging AI Startups in 2025

## Executive Summary
The AI startup landscape in 2025 is characterized by rapid growth and innovation, particularly in sectors such as healthcare, public safety, and enterprise solutions. Startups are increasingly focusing on integrating AI into existing business applications, enhancing operational efficiency, and addressing complex challenges through advanced technologies. The investment in generative AI and AI-driven automation is expected to surge, reflecting a broader trend towards AI adoption across various industries.

## Top Emerging AI Startups

1. **Company name:** AI Squared  
   **Main product or innovation:** Simplifies AI integration into business applications.  
   **Why it matters:** Enables rapid deployment and scaling of AI solutions, enhancing operational efficiency.  
   **Sector:** Enterprise Software  
   **Source:** CRN - [Link](https://www.crn.com/news/ai/2025/the-10-hottest-ai-st

In [9]:
search_results = safe_search("top emerging AI startups 2025 innovations funding products")

summary_prompt = f"""
Use the search results below to create a concise Markdown summary.

Search results:
{search_results}

Structure:
# Emerging AI Startups in 2025

## Executive Summary
Briefly summarise the main trends.

## Top Emerging AI Startups
For each startup, include:
- **Company name:**
- **Main product or innovation:**
- **Why it matters:**
- **Sector:**
- **Source:**

Include 6-8 startups maximum.

## Key Innovation Trends
List 4-5 trends.

## Final Takeaway
Short conclusion.
"""

response_2 = llm.invoke(summary_prompt)
display(Markdown(response_2.content))

# Emerging AI Startups in 2025

## Executive Summary
In 2025, the AI startup landscape is characterized by significant funding and innovation, with a focus on specialized applications and enterprise solutions. A total of 49 AI companies in Greater Seattle alone received funding, reflecting a healthy mix of early-stage and growth-stage investments. Major funding rounds highlight a trend towards AI-native platforms that deliver measurable operational ROI, indicating a maturation of the market.

## Top Emerging AI Startups

- **Company name:** Dipp AI Technologies  
  **Main product or innovation:** Orchestration layer for superintelligent multi-agent enterprise systems.  
  **Why it matters:** Aims to enhance enterprise efficiency through advanced AI coordination.  
  **Sector:** Enterprise AI  
  **Source:** [Greater Seattle](https://greater-seattle.com/ai-startups-funding-2025/)

- **Company name:** Dynamiq  
  **Main product or innovation:** Low-code platform for creating custom AI agents and fine-tuned LLMs.  
  **Why it matters:** Reduces implementation costs and accelerates deployment for enterprises.  
  **Sector:** Enterprise AI  
  **Source:** [Sends](https://sends.co/blog/top-5-emerging-startup-trends-investors-should-watch-in-2025)

- **Company name:** Scalestack  
  **Main product or innovation:** AI-powered go-to-market infrastructure for automating RevOps and SalesOps.  
  **Why it matters:** Enhances revenue growth and operational efficiency for clients.  
  **Sector:** SaaS  
  **Source:** [Sends](https://sends.co/blog/top-5-emerging-startup-trends-investors-should-watch-in-2025)

- **Company name:** InferX  
  **Main product or innovation:** AI Native OS for Inference Platform.  
  **Why it matters:** Provides a robust infrastructure for AI applications.  
  **Sector:** Infrastructure  
  **Source:** [Greater Seattle](https://greater-seattle.com/ai-startups-funding-2025/)

- **Company name:** Cognition AI  
  **Main product or innovation:** AI coding agent.  
  **Why it matters:** Represents a significant advancement in developer tools, enhancing productivity.  
  **Sector:** Developer Tools  
  **Source:** [Bitcoinworld](https://www.binance.com/en/square/post/35305668863545)

- **Company name:** Luma AI  
  **Main product or innovation:** Visual media creation platform.  
  **Why it matters:** Attracts substantial investment, indicating strong market demand for creative AI applications.  
  **Sector:** Creative AI  
  **Source:** [Bitcoinworld](https://www.binance.com/en/square/post/35305668863545)

## Key Innovation Trends
1. **AI-Native Platforms:** Startups are increasingly focusing on AI-native solutions that outperform traditional SaaS in customer retention and operational efficiency.
2. **Specialized Applications:** There is a growing trend towards domain-specific AI tools, such as AI copilots for niche markets.
3. **Increased Funding:** Record-breaking investments in AI startups, particularly in foundational models and generative AI.
4. **Early-Stage Investment Surge:** A significant number of newly founded companies are attracting early-stage funding, indicating a vibrant innovation ecosystem.
5. **Focus on Measurable ROI:** Investors are prioritizing AI solutions that demonstrate clear operational benefits over mere novelty.

## Final Takeaway
The AI startup ecosystem in 2025 is thriving, driven by substantial investments and a focus on specialized, measurable solutions. As the market matures, startups that effectively leverage AI to solve specific challenges are likely to emerge as leaders in their respective sectors.

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [10]:
# Your prompt here

prompt_3 = """
Compare OpenAI ChatGPT Team and Anthropic Claude Pro using recent web search results.

Focus on:
- Key features
- Pricing
- Target users
- Strengths
- Limitations
- Reviews or user feedback where available

Return the answer in clean Markdown.

Structure:

# ChatGPT Team vs Claude Pro

## Executive Summary
Brief comparison in 3-4 sentences.

## Comparison Table
Include columns:
- Product
- Provider
- Main features
- Pricing
- Best for
- Key limitation

## ChatGPT Team
Summarise:
- Features
- Pricing
- Strengths
- Limitations
- Source

## Claude Pro
Summarise:
- Features
- Pricing
- Strengths
- Limitations
- Source

## Reviews and User Feedback
Summarise common themes from available reviews or public feedback.

## Final Recommendation
Explain which product is better for which type of user.
"""

search_results = safe_search(
    "ChatGPT Team pricing features reviews compared with Anthropic Claude Pro pricing features reviews 2025"
)

summary_prompt = f"""
Use the web search results below to answer the prompt.

Prompt:
{prompt_3}

Search results:
{search_results}

Keep the answer concise, factual, and structured in Markdown.
"""

In [11]:
response_3 = llm.invoke(summary_prompt)
display(Markdown(response_3.content))


# ChatGPT Team vs Claude Pro

## Executive Summary
ChatGPT Team and Anthropic Claude Pro are two leading AI tools priced similarly at $20/month for individual users and $30/user/month for team plans. ChatGPT excels in a broad range of integrations and tools, including image generation, while Claude Pro is noted for its strong performance in long-form writing and compliance-sensitive tasks. Both have unique strengths and limitations, making them suitable for different user needs.

## Comparison Table

| Product         | Provider  | Main Features                                                                 | Pricing                     | Best for                          | Key Limitation                          |
|------------------|-----------|-------------------------------------------------------------------------------|-----------------------------|-----------------------------------|-----------------------------------------|
| ChatGPT Team     | OpenAI    | DALL-E image generation, web browsing, 1000+ specialized GPTs, team collaboration | $30/user/month              | Creative professionals, teams needing diverse tools | Limited long-form writing capabilities   |
| Claude Pro       | Anthropic | Strong long-form writing, nuanced reasoning, coding capabilities, compliance focus | $20/month                   | Document-heavy tasks, regulated industries | No image generation, fewer integrations  |

## ChatGPT Team
- **Features**: Includes DALL-E for image generation, web browsing, and access to over 1000 specialized GPTs. Team collaboration features allow for shared projects and workspace management.
- **Pricing**: $30/user/month, with potential discounts for annual billing.
- **Strengths**: Broad toolset, strong in creative tasks, and extensive third-party integrations.
- **Limitations**: Less effective for long-form writing compared to Claude Pro.
- **Source**: [SaaS Price Pulse](https://www.saaspricepulse.com/compare/chatgpt-vs-claude)

## Claude Pro
- **Features**: Focuses on long-form writing quality, nuanced reasoning, and coding capabilities. It supports deep research and has a strong compliance focus.
- **Pricing**: $20/month, with no annual discount.
- **Strengths**: Excels in document-intensive tasks and is preferred in regulated industries for its analytical capabilities.
- **Limitations**: Lacks image generation features and has fewer integrations compared to ChatGPT.
- **Source**: [Zapier](https://zapier.com/blog/claude-vs-chatgpt/)

## Reviews and User Feedback
Common themes from user feedback indicate that ChatGPT Team is favored for its versatility and creative capabilities, making it ideal for teams that require a variety of tools. In contrast, Claude Pro is praised for its writing quality and analytical strengths, particularly in compliance-heavy environments. Users often appreciate the ease of use and the effectiveness of both tools in their respective strengths.

## Final Recommendation
For teams and users who need a versatile AI with a wide range of tools, ChatGPT Team is the better choice. However, for professionals focused on long-form writing, compliance, and document-heavy tasks, Claude Pro is the superior option. Users may find value in utilizing both tools for different aspects of their work.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
